## human written - ai never touches this
### prerequisites and setup
See prerequisites and setup in
`tasks/tasks-20260519-review-231/SPEC.md`

Use `./WORK.md` as
your own workbook for
recording actions you have in mind and
recording in progress and completed, or
any other notes you feel you need.
Write as if for a
busy tech lead and
also to be helpful for the executor, so
lean concise text that 
contains all relevant info inplace but is
focused and very well organized.

### actual task
We train a robust ML model to predict missingness patterns to be able to simulate that on fully complete keys.
We assume MAR for this even thought really it's MNAR but we anticipate that available non-missing fields will substantially correlate with the missing value, and so MAR works.
We use the 307 source keys, with all innerdicts across xlsx, docx, and ssn.
We calculate how many contain at least one missing value and how many none; optional docx fields are still optional; so we're only concerned with fields that are in actual output txt/docx.
Then for all keys we prepare a missingess mask; for nonmissing we of course have all ones; every key within innerdict is considered a data point.

features we use for prediction: country name (categorical), world bank economy (ordinal),  perhaps ktp.priority_label (ordinal) but for this one need to do some factor analysis because cocorrelation can be high and we'd like to know which of the features predicts best, p_gf inference sources and inference_counts - perhaps use both (integer), p_gf per se (float), the various work counts (integer) - all the ones we use in ssn hit logic and also add cited_by_count and don't forget ssn_sum_hit_1pct, also need factor analysis to see which predicts best, and also add all the scientific productivity metrics from ssnau - also to factor analysis, openalex_match (bool).

Inference:
We do go over all nonmissing only (as missing is reserved for heldout) and
based on features predict the missingness mask.
We cache the output, that is, per each nonmissing key we'll have a cached simulated mask.
And so ultimately we have simulated missingness masks for all truly nonmissing.
Then we randomly sample from the simulated missing ones and return.
Implement this in cells below. 
Use commentaries within cells to understand which contains which, but
do not remove or edit the commentaries.

In [ ]:
# here write all code for retrieving and preprocessing (if any) data from db. use `config.repl.json` as source config from which read db (readonly). we partition into missing and nonmissing, nonmissing are reserved for inference and missing are train/test split.

In [ ]:
# here write all code for training and validation, output should be metrics

In [ ]:
# here write inference, output should be dataframe of cached simulated masks

In [ ]:
# here write a sampler that returns a random record from simulated pool - the record must be with mask applied and should be divided into input (all non missing fields in json) and output (missing fields), perhaps all as one json. so ideally you just provide a single outerdict record with source key and innerdicts inside. in think actually input may be the raw outerdict record where the missing is just missing, and output (ground truth) would be exact same record with this value filled in. but how it should work it should actually only provide input and output must be concealed

In [ ]:
# here simulate the use of that sampler where in a loop we request inputs and supply some outputs (that is, only missing keys/values) and the sampler (which is also an evaluator) accepts this and returns pass or fail. and so after loop we get accuracy metrics.

## how ai understood the spec

### delivery boundary and inherited constraints

This notebook is the complete implementation surface for the later task: fill the
five commented code cells in place, preserving each human commentary line verbatim
as the first line of its cell. This task only specifies that implementation; it does
not add or execute model code.

The implementation must never run or import `src.repl`. Read the database path from
`config.repl.json`, open `data/scisci_process.duckdb` with
`duckdb.connect(..., read_only=True)`, and treat the persisted database as the only
data source. Reconstruct records from `outerdict_name_keys`, `xlsx_innerdicts`,
`docx_innerdicts`, and `ssn_innerdicts`; do not read generated TXT, DOCX, ZIP, CSV,
or other data artifacts. The source database must never be attached or reopened
writable. Model/cache artifacts may be written only below this task directory.

The future implementation will require scikit-learn for preprocessing, factor
analysis, regularized classifiers, and metrics. Add it as a declared project/pixi
dependency and refresh the lock file at implementation time; do not install packages
imperatively from a notebook cell.

### modeling unit, target, and mask semantics

One `ktp.source_key` is one independent modeling record. All innerdicts associated
with it remain part of that record; XLSX/SSN rows must not be exploded into
independent train/test observations. Each required target field is one binary label
within the record's multivariate mask. This is how “every key within the innerdict is
a data point” is represented without pseudoreplication or source-key leakage.

The mask is defined as `1 = retain/observed` and `0 = simulate missing/hide`. Its
target universe is the required DOCX `ktp.table_1_*` family used by current step-10
subset semantics, not every nullable column in the three source schemas. Derive the
field list dynamically from the persisted DOCX schema using
`KTP_DOCX_TABLE_1_PREFIX`, excluding `KTP_DOCX_OPTIONAL_EMPTY_COLS`, and use
`KTP_TABLE_1_EMPTY_VALUE_PLACEHOLDERS` through the same non-empty semantics as
`_is_non_empty_value()` in `src/steps/step_10_build_cards.py`. Do not duplicate a
different missing-value rule.

The current required target fields are:

| mask position | field |
|---:|---|
| 0 | `ktp.table_1_academic_position_s_` |
| 1 | `ktp.table_1_age_first_publication_according_to_openalex_profile` |
| 2 | `ktp.table_1_education` |
| 3 | `ktp.table_1_gender` |
| 4 | `ktp.table_1_links_` |
| 5 | `ktp.table_1_place_of_residence` |
| 6 | `ktp.table_1_researcher_author` |
| 7 | `ktp.table_1_social_capital` |

The optional-empty DOCX fields—socioeconomic status,
race/ethnicity/language/culture, topics, footnotes, and comments—remain context and
are never mask targets. Likewise, null/absent XLSX and SSN fields are context
availability, not target missingness. This distinction is necessary because fields
such as `hcr.twitter` and `ssnad.last_known_institution` are structurally empty in
the current source; treating every source null as a target would eliminate the
intended fully complete cohort.

Choose one canonical DOCX row per source key by fewest missing required fields, then
the repository's stable draw/filename/fragment ordering. This preserves the rule
that any complete DOCX row makes the source key complete. If a future source key has
no DOCX row, synthesize an empty canonical target row and mark all required fields
missing. Retain every noncanonical row as record context.

### verified current-data audit

Read-only inspection of the current configured database gives:

| item | current count |
|---|---:|
| source keys | 307 |
| XLSX innerdicts / represented source keys | 2,018 / 307 |
| DOCX innerdicts / represented source keys | 317 / 307 |
| SSN innerdicts / represented source keys | 2,044 / 304 |
| keys whose canonical required-DOCX mask is all ones | 207 |
| keys whose canonical required-DOCX mask contains a zero | 100 |
| distinct non-complete masks | 25 |

The 100 incomplete keys are the only train/test population. Their current field
missing counts are:

| required field | missing keys |
|---|---:|
| education | 71 |
| age at first publication | 53 |
| academic position(s) | 41 |
| gender | 38 |
| links | 27 |
| social capital | 25 |
| place of residence | 20 |
| researcher/author | 0 |

The missing-field count per incomplete key is: 51 keys with one missing field, 10
with two, 7 with three, 4 with four, 7 with five, 15 with six, and 6 with seven.
`ktp.table_1_researcher_author` is currently a constant-observed target and therefore
gets a deterministic all-one predictor unless future data introduces both classes.

The complete 207 keys must not participate in fitting, hyperparameter selection,
calibration, threshold selection, or reported test metrics. They are inference-only.
Consequently, the estimand is the conditional distribution
`P(mask | available features, at least one field is missing)`, not the probability
that an arbitrary key has any missingness. The notebook must state this MAR transfer
assumption and must not claim that it has learned the complete-versus-incomplete
incidence rate.

### source-key feature matrix

Build one deterministic feature row per source key from all of its XLSX and SSN
innerdicts. Never use a DOCX target value, a target-presence indicator, partition or
subset flags, source key/name text, filename, fragment, draw number, or any feature
computed after seeing the mask. Feature preprocessing must be fit on training rows
only and then applied unchanged to validation, test, and inference rows.

Required candidate feature families are:

| family | persisted fields and treatment |
|---|---|
| geography | Parse and union `ktp.hcr_world_bank_economies` across XLSX rows as a multi-valued categorical country feature; retain an explicit unknown/empty level and pool rare levels inside each training fold. |
| World Bank economy | Parse `ktp.hcr_world_bank_economies_income_group`; map `L < LM < UM < H` using repo-owned labels and expose presence flags plus robust lowest/highest ordinal summaries for multi-country keys. |
| priority | Prefer the existing integer `ktp.priority` as ordinal. Keep `ktp.priority_label` for audit/display; do not feed both duplicate encodings unless validation establishes incremental value. |
| p_gf evidence | `ssnau.inference_sources`, `ssnau.inference_counts`, and `ssnau.p_gf`, each with an explicit missing indicator. |
| SSN hit/work evidence | `ktp.ssn_sum_hit_1pct`, `ktp.ssn_hit_works_count_raw`, `ssnad.works_count`, and `ssnad.cited_by_count`. |
| SSN hit flags | `ktp.ssn_hit_sum_hit_1pct_is_tukey_outlier`, `ktp.ssn_hit_works_count_is_tukey_outlier`, `ktp.ssn_hit_cited_by_count_is_tukey_outlier`, `ktp.ssn_hit_row_has_tukey_outlier`, and `ktp.ssn_hit_fallback_no_tukey_outlier`. |
| scientific productivity | All current numeric `ssnau` productivity metrics: `ssnau.avg_c10`, `ssnau.avg_logc10`, `ssnau.productivity`, and `ssnau.h_index`. |
| OpenAlex | `ktp.openalex_match` as a nullable boolean with a separate missing indicator. |

For repeated SSN rows, use every row through source-key summaries rather than an
arbitrary author row. Include the innerdict count; for numeric metrics use median,
IQR, minimum, and maximum, with the raw value naturally recovered when there is one
row. Do not sum author-level productivity/citation values across alternative author
matches. For booleans use any/all/observed proportion plus missingness. For repeated
XLSX rows use set union for categorical values and robust min/max/mode summaries for
ordinals. Keep the resulting feature count controlled because the labeled cohort is
only 100 records.

### collinearity and factor analysis

Factor analysis is an audit and optional representation step, not evidence of which
original feature predicts missingness. On the training partition only:

1. Report distributions, missingness, near-zero variance, Spearman correlations for
   numeric/ordinal features, Cramér's V for categorical pairs, and mutual information
   for mixed pairs.
2. Standardize the numeric count/productivity block and run
   `sklearn.decomposition.FactorAnalysis`; choose the candidate factor count by
   parallel analysis and cross-validated downstream performance, not an arbitrary
   eigenvalue cutoff.
3. Emit a loadings table that makes the expected redundancy among
   `ssnad.works_count`, `ktp.ssn_hit_works_count_raw`, cited counts, hit counts, and
   `ssnau` productivity metrics visible.
4. Compare original regularized features, factor scores, and one representative per
   correlated cluster. Determine predictive value with held-out permutation
   importance and family-level ablation. Factor loadings alone must never be called
   predictive importance.

The selected representation is the simplest one whose validation performance is
not materially worse than the best candidate. This analysis decides whether
`ktp.priority`, both p_gf evidence counts, and redundant work/productivity fields
remain in the final model.

### training, validation, and mask model

Use a fixed seed derived from `config.sample_seed` (currently 42), with separate
labelled sub-seeds for splitting, tuning, final fitting, inference-mask draws, and
sampler order. All split membership is by source key.

Create one locked 20% test set from the 100 incomplete keys using deterministic
iterative multilabel stratification over the seven nonconstant target columns and
missing-count bins. Keep both classes for each target in train/test where feasible;
do not stratify on the 25 exact mask signatures because most are rare. Use only the
remaining 80% for preprocessing choice and nested/repeated cross-validation. Inspect
the locked test set once after all model choices are frozen, then refit the chosen
pipeline on all 100 incomplete keys for inference. Implement the small deterministic
greedy multilabel splitter in the notebook rather than adding another dependency.

Fit a probabilistic binary model for each nonconstant field. The primary small-data
model is class-weighted elastic-net logistic regression over fold-local encoded and
scaled features. Compare it with a conservatively tuned shallow nonlinear
scikit-learn tree/boosting model; do not run an unconstrained hyperparameter search.
Calibrate probabilities from out-of-fold predictions using sigmoid calibration when
there are enough examples. A constant target uses its empirical constant.

Compare against two mandatory baselines: per-field training prevalence and
feature-independent empirical resampling of complete observed mask patterns. A
feature-dependent model is accepted only when locked-test proper scores and repeated
cross-validation are at least as good as the empirical baseline within uncertainty;
otherwise use the simpler baseline and report that result rather than asserting a
spurious predictor effect.

Independent field probabilities must not be sampled independently because that can
create implausible combinations. Preserve joint missingness by limiting generated
masks to the empirical non-complete mask support. For source-key features `x`, let
`q_j(x)` be the calibrated probability that field `j` is missing. Give each observed
mask `s` a Dirichlet-smoothed empirical prior and reweight it by the Bernoulli
likelihood across fields:

`weight(s | x) ∝ (count(s) + alpha) × product_j q_j(x)^(1-s_j) × (1-q_j(x))^s_j`.

Normalize these weights and draw one supported non-complete mask. This models
feature-dependent pattern choice while retaining real co-missing structures. The
all-one mask is excluded because inference is explicitly simulating a missing case
conditional on missingness.

Validation output must include per-field prevalence, support, ROC-AUC and average
precision when defined, log loss, Brier score, and F1 with thresholds chosen only on
training folds. Also report micro/macro F1, Hamming loss, Jaccard score, exact-mask
accuracy, missing-count MAE, and calibration. For sampled joint masks, compare
held-out field marginals, missing-count distribution, mask frequencies, and pairwise
co-missing correlations. Include bootstrap confidence intervals across source keys.

Because inference transfers from 100 incomplete to 207 complete records, report a
feature-shift audit: category coverage/unknown rates, standardized numeric
differences, out-of-training-range counts, and model uncertainty. Flag unsupported
inference rows in the cache; do not silently extrapolate or use complete-record
labels to tune the model.

### code-cell contracts

#### Retrieval and preprocessing cell

After its existing commentary, this cell must:

- load config and open one explicitly read-only DuckDB connection;
- load and JSONL-parse the four persisted relations, verify source-key uniqueness
  and membership, and reconstruct stable ordered outerdict records;
- derive required fields, canonical DOCX rows, masks, and the 100/207 partition;
- print the current cohort, innerdict, field-missing, missing-count, and mask-pattern
  audits, failing loudly if invariants drift;
- build the source-key feature matrix from all XLSX/SSN rows; and
- close the connection after materializing the in-memory source snapshot.

Its main outputs are the ordered outerdict mapping, target-field list, canonical-row
metadata, source-key feature frame, 100-row labeled frame, and 207-row inference
frame.

#### Training and validation cell

After its commentary, this cell performs the fold-local collinearity/factor audit,
locked train/test split, baseline comparisons, model selection, calibration, and
metrics above. It then refits the frozen pipeline on all 100 incomplete keys. Display
compact metrics, factor loadings/correlation clusters, selected feature families,
test predictions, and transfer diagnostics; do not merely print a training score.

#### Inference and cache cell

After its commentary, this cell computes field probabilities for all 207 complete
keys and draws one empirical-support mask per key. Sampling must be stable under
input row reordering by deriving a per-key RNG seed from a cryptographic hash of the
global seed, source fingerprint, model version, and source key.

The displayed `simulated_masks_df` has one row per complete source key and at least:

- `ktp.source_key` and canonical DOCX row locator;
- one boolean `mask.<required-field>` column per target;
- one `p_missing.<required-field>` column per target;
- ordered `missing_fields`, `missing_count`, empirical pattern ID, and
  `has_simulated_missingness`;
- out-of-distribution/unsupported flags; and
- source fingerprint, feature-schema hash, model hash/version, seed, and creation
  timestamp.

Cache this frame plus a small manifest under
`tasks/tasks-20260725-mar-mask/cache/`, never in the source database. Cache reuse is
allowed only when the source relation contents/schemas, required-field order,
feature schema, model settings, dependency versions, and seed fingerprints match.
Use an atomic temporary-file replacement so interruption cannot leave a valid-looking
partial cache.

#### Sampler/evaluator cell

After its commentary, this cell defines a stateful seeded sampler over cached rows
with at least one zero. Draw without replacement until exhaustion; a reset starts a
new explicitly seeded epoch. The public sample payload is structured JSON:

```json
{
  "sample_id": "opaque id",
  "source_key": "serialized ktp.source_key",
  "missing_fields": ["ordered required field names"],
  "input": {"source_key": "...", "innerdicts": [{"ktp.filename": "..."}]}
}
```

Reconstruct `input` from all XLSX, DOCX, and SSN innerdicts using final-card order.
Keep `ktp.filename` as provenance/heading, retain JSON-safe non-null card-visible
values, and apply the same technical exclusions as card creation
(`ktp.source_key`, CSV/DOCX row indices, and `ktp.docx_fragment` inside
innerdicts). Remove each masked target field from every DOCX innerdict copy so a
noncanonical duplicate cannot leak it. Do not mutate the stored source record.

The canonical complete DOCX values for exactly `missing_fields` form the private
answer key. Neither the sampler payload, object representation, logs, exceptions,
nor evaluation response may expose those values. A submission contains only
`sample_id` plus a mapping whose key set exactly equals `missing_fields`; reject
missing/extra keys and empty answers as invalid without scoring them.

Evaluate values after deterministic Unicode normalization, line-ending
normalization, trim, and internal-whitespace collapse, while retaining the raw
submission for audit. Accept at most one immutable scored submission per sample and
return only overall pass/fail; retain per-field booleans privately for aggregate
metrics. Never return ground-truth values. Repeated evaluation of one sample ID is
idempotent and cannot change or double-count the first valid submission.

#### Sampler simulation cell

After its commentary, this cell runs a deterministic loop that requests samples,
passes only missing-field/value mappings from a solver callback, records valid,
correct, and incorrect responses, and prints aggregate results. The test harness may
use a clearly private oracle solely to generate a known mixture of correct and
incorrect submissions; the public sampler/evaluator interface must remain concealed.

Report records requested/scored, invalid submissions, record exact-match accuracy,
field-level micro accuracy, and per-field correct/incorrect/support counts. Assert
that no payload or response contains a concealed value and that sampler exhaustion,
reset, idempotent resubmission, and wrong-key rejection behave as specified.

### reproducibility and acceptance checks

The completed notebook is acceptable when all of the following hold:

- every source DB connection is demonstrably read-only and `src.repl` is never used;
- all 307 keys reconcile exactly once into the 100 labeled and 207 inference cohorts
  under the current database, with the target list derived rather than copied;
- no source key crosses a split and no complete key influences fitting/model choice;
- factor analysis, feature importance, and predictive conclusions are clearly
  distinguished;
- reported metrics include held-out baselines, calibration, joint-mask fidelity,
  and inference feature-shift diagnostics;
- inference produces exactly one deterministic cached non-complete mask for every
  complete key and invalidates stale caches;
- sampler inputs preserve full card-visible context while removing masked values
  from every possible DOCX copy;
- hidden outputs never cross the public sampler/evaluator boundary; and
- rerunning with identical source/config/dependencies yields identical splits,
  models, masks, sampler order, and metrics, while a changed seed changes stochastic
  choices but not cohort membership or preprocessing semantics.